In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Employees


In [0]:
df_emp = spark.read.table('project_cata.bronze.employees')
# df_emp.printSchema()
# df_nulls = df_emp.select([sum(when(col(i).isNull() | (trim(col(i)) == ""), 1).otherwise(0)).alias(i) for i in df_emp.columns])
# display(df_nulls)

# converting date to valid date formats
formats = ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy"]
df_emp = df_emp.withColumn('date_of_birth', coalesce(*(try_to_date(col('date_of_birth'), f) for f in formats))).withColumn('hire_date', coalesce(*(try_to_date(col('hire_date'), f) for f in formats))).withColumn('last_updated', to_timestamp(col("last_updated"), "yyyy-MM-dd HH:mm:ss"))

#removing duplicates
df_emp = df_emp.withColumn('rono', row_number().over(Window.partitionBy('employee_id').orderBy(col('last_updated').desc())))
df_emp = df_emp.filter(col('rono') == 1).drop('rono')
# df_dups = df_emp.groupBy('employee_id').agg(count('employee_id').alias('ecount')).filter(col('ecount')>1)
# display(df_dups)

#Replacing invalid emails with Nulls
eregex = r'^[A-Za-z0-9_.-]+@[A-Za-z0-9_.-]+\.[A-Za-z0-9_]{2,}$'
df_emp = df_emp.withColumn('email', when(col('email').rlike(eregex), col('email')).otherwise(None))
# display(df_emp.filter(col('email').isNull()))

#checking any employees without department
df_dep = spark.read.table('project_cata.bronze.departments')
df_dep = df_dep.withColumn('sub_departments', explode(col('sub_departments.department'))).withColumn('sub_dep_id', col('sub_departments._id')).withColumn('sub_dep_name', col('sub_departments.name'))

df_emp = df_emp.join(df_dep.select('sub_dep_id'), df_emp.department_id == df_dep.sub_dep_id, 'left').withColumn('reject_reason', when(col('sub_dep_id').isNull(), 'No Dep Id').when(col('hire_date').isNull(), 'No hire date').otherwise(None)).drop('sub_dep_id')

#Converting salry to float
df_emp = df_emp.withColumn('base_salary', round( regexp_replace(col('base_salary'), ',' , '').try_cast('float'), 2))

quatantine = df_emp.filter(col('reject_reason').isNotNull())
df_emp = df_emp.filter(col('reject_reason').isNull())

# Insert data to silver Employees
from delta.tables import DeltaTable

if not spark.catalog.tableExists('project_cata.silver.employees'):
    df_emp.write.format('delta').saveAsTable('project_cata.silver.employees')
else:
    target = DeltaTable.forName(spark , 'project_cata.silver.employees')
    target.alias("t").merge(df_emp.alias("s"), 't.employee_id=s.employee_id')\
        .whenMatchedUpdateAll(condition='s.last_updated >= t.last_updated ')\
            .whenNotMatchedInsertAll()\
                .execute()

## Departments

In [0]:
df_dep = spark.read.table('project_cata.bronze.departments')

#Flattern the JSON
df_parent_dep = df_dep.select(col('_id').alias('department_id'), col('name').alias('department_name'), col('_cost_center').alias('cost_center'), lit(None).alias('parent_department_id'))
# display(df_dep)

df_child_dep = df_dep.select(col('_id').alias('parent_department_id'), explode(col('sub_departments.department')).alias('d'))\
    .select(col('parent_department_id'), col('d._cost_center').alias('cost_center'), col('d._id').alias('department_id'), col('d.name').alias('department_name'))

df_dep = df_parent_dep.unionByName(df_child_dep)

df_dep = df_dep.filter((col('department_id').isNotNull()) & (col('parent_department_id').isNull() | (col('department_id') != col('parent_department_id'))) & (col('cost_center').isNotNull()))

df_dep.write.format('delta').mode('overwrite').saveAsTable('project_cata.silver.departments')

## Job Roles

In [0]:
df_job_roles = spark.read.table('project_cata.bronze.job_roles')

df_job_roles = df_job_roles.select(col('_id').alias('job_role_id'), col('title').alias('job_title'), col('job_family'), col('_grade').alias('grade'), col('_hourly').alias('is_hourly'), col('department_id'), col('people_manager').alias('is_people_manager'), col('salary_band.min').alias('salary_min'), col('salary_band.max').alias('salary_max'), col('salary_band._currency').alias('salary_currency'))

dupes = df_job_roles.groupBy('job_role_id').agg(count('job_role_id').alias('count')).filter(col('count')>1)

if dupes.count() > 1:
    df_job_roles.withColumn('ccount', rank('job_role_id').over(Window.partitionBy('job_role_id'))).filter(col('ccount')==1).drop('ccount')

df_job_roles = df_job_roles.withColumn('is_hourly', when(((col('is_hourly') == 'Y') | (col('is_hourly') == 'true')) , True).when(((col('is_hourly') == 'N') | (col('is_hourly') == 'false')) , False)).filter((col('salary_min') > 0) & (col('salary_max') > col('salary_min')))

df_dep = spark.read.table('project_cata.silver.departments')
df_job_roles = df_job_roles.join(df_dep , on=['department_id'] , how='left').filter(col('department_name').isNotNull())\
    .drop('department_name', 'cost_center', 'parent_department_id')

df_job_roles.write.format('delta').mode('overwrite').saveAsTable('project_cata.silver.job_roles')


## Locations

In [0]:
df_locations = spark.read.table('project_cata.bronze.locations')

df_locations = df_locations.filter(col('name').isNotNull()).select(col('_id').alias('location_id'), col('name').alias('location_name'), col('_status').alias('status'), col('address.city').alias('city'), col('address.country_code').alias('country_code'), col('address.region').alias('region'), col('facility._type').alias('facility_type'), col('facility.headcount_capacity').alias('headcount_capacity'), col('facility.timezone').alias('timezone'), col('is_manufacturing_site'), col('local_currency')).distinct()

df_locations.write.format('delta').mode('overwrite').saveAsTable('project_cata.silver.locations')


## Employee Changes

In [0]:
df_emp_changes = spark.read.table('project_cata.bronze.employee_changes')

#de dup change id
df_emp_changes = df_emp_changes.withColumn('rono', row_number().over(Window.partitionBy('change_id').orderBy('change_timestamp'))).filter(col('rono')<=1).drop('rank').drop('rono')

#replace ""/"null" with Null and format date
def null_if_blank(c):
    return when((lower(trim(col(c))) == '') | (lower(trim(col(c))) == 'null'), None).otherwise(trim(col(c)))

def to_date_muliti(c):
    formats = ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy"]
    return coalesce(*[try_to_date(trim(col(c)), d) for d in formats])

def clean_salary(c):
    # return expr(f"try_cast(replace(trim({c}), ',', '') as decimal(18,2))")
    return regexp_replace(trim(col(c)), ',', '').try_cast('decimal(18,2)')

df_emp_changes = df_emp_changes.withColumn('change_type', null_if_blank('change_type'))\
    .withColumn('currency', null_if_blank('currency'))\
        .withColumn('employment_type', null_if_blank('employment_type'))\
            .withColumn('effective_date', to_date_muliti('effective_date'))\
                .withColumn('change_timestamp', try_to_timestamp('change_timestamp'))\
                    .withColumn('base_salary', clean_salary('base_salary'))

dupes = df_emp_changes.groupBy('employee_id', 'effective_date').agg(count('employee_id').alias('ecount')).filter(col('ecount')>1)

if dupes.count() > 1:
    df_emp_changes = df_emp_changes.withColumn('rono', row_number().over(Window.partitionBy('employee_id', 'effective_date').orderBy(col('effective_date').desc()))).filter(col('rono')==1).drop('rono')

#Implement sdc2
w = Window.partitionBy('employee_id').orderBy('effective_date', 'change_timestamp')
max_date = lit("9999-12-31").try_cast('date')

df_emp_changes = df_emp_changes.withColumn('start_date', col('effective_date'))\
                                .withColumn('next_start_date', lead(col('effective_date')).over(w))\
                                    .withColumn('end_date', when(col('next_start_date').isNotNull(), date_sub(col('next_start_date'), 1)).otherwise(max_date))\
                                        .withColumn('end_date', when(col('change_type') == 'TERMINATION', col('effective_date')).otherwise(col('end_date'))).drop('next_start_date')
                

#write into silver
df_emp_changes.write.format('delta').mode('append').saveAsTable('project_cata.silver.employee_history')

## Payroll

In [0]:
df_payroll = spark.read.table('project_cata.bronze.payroll')

amount_cols = ["base_pay","overtime_pay","bonus","gross_pay",
               "income_tax","social_contribution","pension_employee","net_pay"]

def change_to_money(c):
    return regexp_replace(col(c), ',', '').try_cast('decimal(18, 2)')

for c in amount_cols:
    df_payroll = df_payroll.withColumn(c, change_to_money(c))

df_payroll = df_payroll.withColumn('pay_period', try_to_date(concat(trim(col('pay_period')),lit('-1'))))\
    .withColumn('pay_date', try_to_date('pay_date'))\
        .withColumn('currency', when(((col('currency') == '') | (col('currency').isNull())), None).otherwise(col('currency')))

w = Window.partitionBy("payroll_id").orderBy(
        when(upper("payroll_run_id").contains("RERUN"), 1).otherwise(0).desc(),
        col("payroll_run_id").desc())

df_payroll = df_payroll.withColumn("rn", row_number().over(w)).filter(col('rn') == 1).drop('rn')

df_payroll.write.format('delta').mode('append').saveAsTable('project_cata.silver.payroll')


## Attendance

In [0]:
df_attendance = spark.read.table('project_cata.bronze.attendance')
emp_df = spark.read.table('project_cata.silver.employees')
loc_df = spark.read.table('project_cata.silver.locations')

# replace invalid values with null
PLACEHOLDERS = ["", "null", "-", "--", "n/a", "na", "none", "nan"]
check_nulls = lambda c: lower(trim(col(c))).isin(*PLACEHOLDERS)
for c in ['attendance_id', 'employee_id', 'shift', 'status', 'location_id', 'source']:
    df_attendance = df_attendance.withColumn(c, when(check_nulls(c), None).otherwise(col(c)))

# format date and timestamp and decimal
df_attendance = df_attendance.withColumn('work_date', try_to_date(col('work_date')))\
    .withColumn('clock_in', try_to_timestamp(col('clock_in')))\
        .withColumn('clock_out', try_to_timestamp(col('clock_out')))\
            .withColumn('overtime_hours',  expr("try_cast(overtime_hours as decimal(5,2))"))\
                .withColumn('shift', initcap(trim(col('shift'))))\
                    .withColumn('status', initcap(trim(col('status'))))

#Transformations
df_attendance = df_attendance.withColumn('work_hours', ((unix_timestamp(col('clock_out')) - unix_timestamp(col('clock_in')))/3600).try_cast('decimal(5,2)'))

df_attendance = df_attendance.withColumn('rono', row_number().over(Window.partitionBy('attendance_id').orderBy(col('attendance_id').asc()))).filter(col('rono')==1).drop('rono')


# display(df_attendance.select([sum(col(c).isNull().cast('int')).alias(c) for c in ['attendance_id', 'employee_id', 'work_date']]))
for c in ['attendance_id', 'employee_id', 'work_date']:
    df_attendance = df_attendance.withColumn(c, when(col(c).isNotNull(), col(c)))

df_attendance = df_attendance.withColumn('overtime_hours', when(col('overtime_hours') > 0 , col('overtime_hours')))

df_attendance = df_attendance.join(emp_df, df_attendance.employee_id == emp_df.employee_id, 'left_semi')
df_attendance = df_attendance.join(loc_df, df_attendance.location_id == loc_df.location_id, 'left_semi')

df_attendance = df_attendance.filter(~(col('clock_in') > col('clock_out')))

df_attendance.write.format('delta').mode('append').saveAsTable('project_cata.silver.attendance')